# 6. FEATURE ENGINEERING

In [15]:
import pandas as pd
import numpy as np
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.data_loader import load_and_merge_data

df, water_quality, landsat, terraclimate = load_and_merge_data()

print(df.shape)
df.head()

(9319, 16)


,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,month,year,dayofyear,nir,green,swir16,swir22,NDMI,MNDWI,pet
0,-28.760833,17.730278,2011-01-02,128.912,555.0,10.0,1,2011,2,11190.0,11426.0,7687.5,7645.0,0.185538,0.195595,174.2
1,-26.861111,28.884722,2011-01-03,74.720,162.9,163.0,1,2011,3,17658.5,9550.0,13746.5,10574.0,0.124566,-0.180134,124.1
2,-26.450000,28.085833,2011-01-03,89.254,573.0,80.0,1,2011,3,15210.0,10720.0,17974.0,14201.0,-0.083293,-0.252805,127.5
3,-27.671111,27.236944,2011-01-03,82.000,203.6,101.0,1,2011,3,14887.0,10943.0,13522.0,11403.0,0.048048,-0.105416,129.7
4,-27.356667,27.286389,2011-01-03,56.100,145.1,151.0,1,2011,3,16828.5,9502.5,12665.5,9643.0,0.141147,-0.142683,129.2


In [16]:
eps = 1e-6

In [17]:
# =========================
# Existing core features
# =========================
df["nir_swir16_ratio"] = df["nir"] / (df["swir16"] + eps)
df["nir_swir22_ratio"] = df["nir"] / (df["swir22"] + eps)
df["green_nir_ratio"] = df["green"] / (df["nir"] + eps)

df["nir_minus_swir16"] = df["nir"] - df["swir16"]
df["nir_minus_green"] = df["nir"] - df["green"]

df["ndmi_pet"] = df["NDMI"] * df["pet"]
df["mndwi_pet"] = df["MNDWI"] * df["pet"]

df["swir_ratio"] = df["swir16"] / (df["swir22"] + eps)

# =========================
# New physical features
# =========================
df["nir_swir_diff"] = df["nir"] - df["swir22"]
df["swir_diff"] = df["swir16"] - df["swir22"]

df["swir22_green_ratio"] = df["swir22"] / (df["green"] + eps)
df["nir_green_ratio"] = df["nir"] / (df["green"] + eps)

df["ndmi_mndwi"] = df["NDMI"] * df["MNDWI"]

df["water_index"] = (df["green"] - df["swir16"]) / (df["green"] + df["swir16"] + eps)
df["turbidity_proxy"] = df["swir22"] / (df["nir"] + eps)

df["nir_pet"] = df["nir"] * df["pet"]
df["swir16_pet"] = df["swir16"] * df["pet"]

df["ndmi_day"] = df["NDMI"] * df["dayofyear"]
df["pet_day"] = df["pet"] * df["dayofyear"]

In [18]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(df.median(numeric_only=True), inplace=True)

In [19]:
targets = [
    "Total Alkalinity",
    "Electrical Conductance",
    "Dissolved Reactive Phosphorus"
]

print("Shape final del dataframe:", df.shape)
print("Missing values totales:", df.isna().sum().sum())
print("Número total de columnas:", len(df.columns))
print("Número de features nuevas creadas:", len(df.columns) - 16)

df.head()

Shape final del dataframe: (9319, 35)
Missing values totales: 0
Número total de columnas: 35
Número de features nuevas creadas: 19


,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,month,year,dayofyear,nir,...,swir_diff,swir22_green_ratio,nir_green_ratio,ndmi_mndwi,water_index,turbidity_proxy,nir_pet,swir16_pet,ndmi_day,pet_day
0,-28.760833,17.730278,2011-01-02,128.912,555.0,10.0,1,2011,2,11190.0,...,42.5,0.669088,0.979345,0.036290,0.195595,0.683199,1949298.00,1339162.50,0.371077,348.4
1,-26.861111,28.884722,2011-01-03,74.720,162.9,163.0,1,2011,3,17658.5,...,3172.5,1.107225,1.849058,-0.022439,-0.180134,0.598805,2191419.85,1705940.65,0.373698,372.3
2,-26.450000,28.085833,2011-01-03,89.254,573.0,80.0,1,2011,3,15210.0,...,3773.0,1.324720,1.418843,0.021057,-0.252805,0.933662,1939275.00,2291685.00,-0.249879,382.5
3,-27.671111,27.236944,2011-01-03,82.000,203.6,101.0,1,2011,3,14887.0,...,2119.0,1.042036,1.360413,-0.005065,-0.105416,0.765970,1930843.90,1753803.40,0.144144,389.1
4,-27.356667,27.286389,2011-01-03,56.100,145.1,151.0,1,2011,3,16828.5,...,3022.5,1.014786,1.770955,-0.020139,-0.142683,0.573016,2174242.20,1636382.60,0.423442,387.6


In [20]:
df.columns.tolist()

['Latitude',
 'Longitude',
 'Sample Date',
 'Total Alkalinity',
 'Electrical Conductance',
 'Dissolved Reactive Phosphorus',
 'month',
 'year',
 'dayofyear',
 'nir',
 'green',
 'swir16',
 'swir22',
 'NDMI',
 'MNDWI',
 'pet',
 'nir_swir16_ratio',
 'nir_swir22_ratio',
 'green_nir_ratio',
 'nir_minus_swir16',
 'nir_minus_green',
 'ndmi_pet',
 'mndwi_pet',
 'swir_ratio',
 'nir_swir_diff',
 'swir_diff',
 'swir22_green_ratio',
 'nir_green_ratio',
 'ndmi_mndwi',
 'water_index',
 'turbidity_proxy',
 'nir_pet',
 'swir16_pet',
 'ndmi_day',
 'pet_day']

In [21]:
df.to_csv("../data/processed_features_v2.csv", index=False)
print("Archivo exportado: ../data/processed_features_v2.csv")

Archivo exportado: ../data/processed_features_v2.csv
